# Exercise 1 – Unlabelled data outside the Olshevsky window

**Companion notebook:** `01_dataset_creation_and_cleanup.ipynb`

Work through the cells in order. Each exercise has a **Problem** statement in markdown followed by a code cell with `# YOUR CODE HERE`. The patterns you need are all in the companion notebook — re-read it if you get stuck.


In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import xarray as xr
import seaborn as sns
from scipy.interpolate import interp1d

sns.set_theme()

data_path = Path('./data')
data_path.mkdir(exist_ok=True)
figure_path = Path('./figures')
figure_path.mkdir(exist_ok=True)

---

# Exercise 1 – Unlabelled data outside the Olshevsky window

## Background

The companion notebook built a dataset for **November–December 2017** using the `label_source='Olshevsky'` option, which attaches human-verified plasma-region labels to each timestep. Those labels exist for a finite time range; outside that window the API will return no labels.

This is a common real-world situation: you have a well-curated labelled set for training, and a much larger *unlabelled* pool that a model should eventually generalise to. Before you can use the unlabelled data, you still need to perform the same quality checks — the energy-grid artefact does not disappear just because labels are absent.

In this exercise you will download a fresh interval, apply the same cleaning pipeline from the companion notebook, and reflect on what changes when labels are not available.

**Suggested interval:** `["2018-03-01", "2018-03-31"]` — after the labeled datarage but still dayside. Feel free to try a different month if you are curious.

We will also download the spacecraft **position** (ephemeris) using `cdasws` and plot the orbit in GSE coordinates — colour-coded by time and optionally annotated with magnetopause and bow shock models. Without region labels, the orbital context can be a informative sanity check.

### Exercise 1.1 – Download the raw data (no labels)

Use `spacephyml`'s `create_dataset` to download one month of MMS-1 omni-directional ion spectra for a time range that falls **outside** the Olshevsky labelled period. Pass `label_source='Unlabeled'` so no labels are requested.

After loading with `xarray`, print the dataset summary and confirm that no `label` variable is present.

In [ ]:
from spacephyml.datasets.creator import create_dataset

# Choose a month outside the Olshevsky label window (Nov–Dec 2017)
TRANGE = ["2018-03-01", "2018-03-31"]

raw_unlabelled_path = data_path / 'ex1_raw_unlabelled.nc'

# YOUR CODE HERE
# 1. Call create_dataset for TRANGE, requesting only the ion energy spectrum.
#    Skip the download if the file already exists.
# 2. Open the file with xarray and store it in a variable called `ds_ex1`.
# 3. Print ds_ex1 and confirm that no label variable appears.


### Exercise 1.2 – Diagnose the energy-grid artefact

Repeat the energy-grid diagnostic from Section 3 of the companion notebook on your new dataset:

1. Plot the histogram of channel-0 energy values. How many peaks do you see? Is the pattern the same as in 2017?
2. Print the unique values for each of the 32 energy bins.
3. Based on the histogram, identify which grid is dominant in this new interval and which is the minority.

In [ ]:
# YOUR CODE HERE
# Plot the histogram of channel-0 energy values for ds_ex1.
# Label axes and add a title. Save the figure to figure_path.


In [ ]:
# YOUR CODE HERE
# Print the unique energy values per bin, as in the companion notebook.


### Exercise 1.2b – Plot the MMS orbit in GSE coordinates

Knowing *where* the spacecraft was is one of the most useful sanity checks you have when labels are absent. If most of the interval is spent deep in the magnetosphere, you should not expect much solar wind data — and a classifier that has never seen magnetospheric crossings will struggle to generalise.

Here we use **cdasws** to download the MMS-1 definitive ephemeris (position in GSE coordinates) and plot the orbit colour-coded by time. As an optional extension, you can overlay a simple Shue et al. (1998) magnetopause model or a Fairfield (1971) bow shock to put the trajectory in physical context.

The starter code below downloads the position data — your job is to complete the orbit plot and optionally add a boundary model.


In [ ]:
from cdasws import CdasWs
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

cdas = CdasWs()

# ── Download MMS-1 definitive ephemeris (GSE position) ─────────────────────
# Dataset : MMS1_MEC_SRVY_L2_EPHT89Q  (MEC = Magnetic Ephemeris & Coordinates)
# Variable: mms1_mec_r_gse            (spacecraft position in GSE, units: km)
status, data = cdas.get_data(
    'MMS1_MEC_SRVY_L2_EPHT89Q',
    ['mms1_mec_r_gse'],
    TRANGE[0], TRANGE[1],          # reuse the time range from Exercise 1.1
)

# Convert position from km → Earth radii (1 R_E = 6371.2 km)
R_E = 6371.2
pos_gse = data['mms1_mec_r_gse']   # shape: (N, 3)  [x, y, z]
pos_time = data['Epoch']            # numpy datetime64 array

x_gse = pos_gse[:, 0] / R_E
y_gse = pos_gse[:, 1] / R_E
z_gse = pos_gse[:, 2] / R_E

# Map time to a [0, 1] scalar for the colour scale
t_num = (pos_time - pos_time[0]) / (pos_time[-1] - pos_time[0])

# ── YOUR CODE HERE ─────────────────────────────────────────────────────────
# Create a figure with two panels: XY and XZ projections.
#
# For each panel:
#   1. Scatter-plot the orbit coloured by time (use t_num as the colour value
#      and a sequential colormap, e.g. 'plasma' or 'viridis').
#   2. Add a colourbar labelled with the start and end date.
#   3. Label axes in units of R_E.
#   4. Mark the Earth with a filled circle at the origin (radius ≈ 1 R_E).
#
# ── OPTIONAL EXTENSION ─────────────────────────────────────────────────────
# Overlay a simple boundary model in the XY (equatorial) plane.
#
# Shue et al. (1998) magnetopause — standoff distance r0 and flaring α:
#   r = r0 * (2 / (1 + cos(theta))) ** alpha
# Typical quiet-time values: r0 = 10.22 R_E, alpha = 0.81
#
# Jelínek et al. (2012) / Fairfield (1971) bow shock — a simple conic:
#   r = L / (1 + eps * cos(theta))
# Typical values: L = 23.3 R_E, eps = 1.16
#
# theta runs from 0 (sunward, +X) to pi (tailward, -X).
# To convert polar → Cartesian:  x = r*cos(theta),  y = r*sin(theta)
#
# theta = np.linspace(0, np.pi, 300)
# r0_mp, alpha_mp = 10.22, 0.81
# r_mp = r0_mp * (2 / (1 + np.cos(theta))) ** alpha_mp
# ax.plot(r_mp * np.cos(theta), r_mp * np.sin(theta), 'k--', label='Magnetopause')
#
# Add a legend and save the figure to figure_path.


### Exercise 1.3 – Apply Strategy A and save the cleaned dataset

Apply the drop strategy (Strategy A from Section 4.1 of the companion notebook) to your unlabelled dataset:

1. Identify the dominant grid using the channel-0 mode.
2. Create a boolean mask and drop the minority-grid timesteps.
3. Print the number of timesteps kept and the fraction retained.
4. Save the cleaned dataset to `data/ex1_cleaned_unlabelled.nc`.

In [ ]:
# YOUR CODE HERE
# Apply Strategy A to ds_ex1 and save the result.


### Exercise 1.4 – What changes without labels? *(Reflection)*

In the companion notebook, Section 5 checked whether the drop strategy introduced **label bias**. That check requires knowing the label of each timestep.

Answer the following questions in the markdown cell below (edit it directly):

1. What bias check did we do in the companion notebook, and why was it important?
2. Without labels, can you perform an equivalent check? What *proxy* information (if any) could you use instead?
3. What does this tell you about the relationship between data quality and the availability of labels?

*(Double-click to edit — write your answers here)*

1. 

2. 

3. 

---

## Summary

In this exercise you reproduced the full dataset-creation pipeline for a time range **outside** the Olshevsky labelled window:

- The same energy-grid artefact appears regardless of whether labels are available — it is a property of the mission, not the labelled subset.
- Strategy A (drop the minority grid) can still be applied, but the bias check from the companion notebook requires a proxy when labels are absent (e.g. orbital position or season).
- This mirrors a common real-world situation: a small curated labelled set for training, and a much larger unlabelled pool to which the model must generalise.

**Key principle:** Always inspect the data — histogram, orbit plot, cadence — before trusting it, labelled or not.

### Further reading
- Olshevsky et al. — labelled MMS dataset: https://doi.org/10.5281/zenodo.17152371
- Roberts et al. (2021) — FPI solar wind mode: https://doi.org/10.1029/2021JA029784
